In [1]:
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from keras_tuner import RandomSearch

# Load and preprocess the IMDb dataset
max_features = 10000  # Vocabulary size
max_len = 200  # Maximum length of input sequences

(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=max_features)
x_train = pad_sequences(x_train, maxlen=max_len)
x_test = pad_sequences(x_test, maxlen=max_len)

# Define the model using hyperparameters and constant variables
def build_model(hp, vocab_size, input_length):
    model = Sequential()
    model.add(Embedding(
        input_dim=vocab_size,  # Passing the constant vocab_size
        output_dim=hp.Choice('embedding_dim', values=[32, 64, 128]),
        input_length=input_length  # Passing the constant input_length
    ))
    model.add(LSTM(
        units=hp.Int('lstm_units', min_value=32, max_value=128, step=32),
        dropout=hp.Float('dropout_rate', min_value=0.2, max_value=0.5, step=0.1)
    ))
    model.add(Dense(1, activation='sigmoid'))
    
    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            hp.Float('learning_rate', min_value=1e-4, max_value=1e-2, sampling='log')),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    
    return model

# Initialize the Random Search tuner
tuner = RandomSearch(
    lambda hp: build_model(hp, vocab_size=max_features, input_length=max_len),
    objective='val_accuracy',
    max_trials=10,
    executions_per_trial=1,
    directory='embedding_tuner',
    project_name='embedding_tuning'
)

# Perform the search
tuner.search(x_train, y_train, epochs=10, validation_data=(x_test, y_test))

# Get the best model
best_model = tuner.get_best_models(num_models=1)[0]
evaluation = best_model.evaluate(x_test, y_test)

print(f"Best model evaluation: {evaluation}")


2024-08-23 00:05:29.009148: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2024-08-23 00:05:29.009478: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2024-08-23 00:05:29.011678: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2024-08-23 00:05:29.018580: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-08-23 00:05:29.029789: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been 

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 5s 0us/step


/home/admin/CODE_WORKSPACE/capstonenlp/SICCapstone_NLP/.venv/lib/python3.11/site-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(



Search: Running Trial #1

Value             |Best Value So Far |Hyperparameter
64                |64                |embedding_dim
32                |32                |lstm_units
0.3               |0.3               |dropout_rate
0.0060809         |0.0060809         |learning_rate

Epoch 1/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 28s 34ms/step - accuracy: 0.6882 - loss: 0.5666 - val_accuracy: 0.8635 - val_loss: 0.3270
Epoch 2/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 27s 35ms/step - accuracy: 0.8915 - loss: 0.2632 - val_accuracy: 0.8712 - val_loss: 0.3022
Epoch 3/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 28s 36ms/step - accuracy: 0.9437 - loss: 0.1511 - val_accuracy: 0.8734 - val_loss: 0.3417
Epoch 4/10
764/782 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - accuracy: 0.9684 - loss: 0.0924

KeyboardInterrupt: 